# Session 8 · Predicting age from sleep EEG (machine learning)

Sleep changes with age: less N3, more awakenings, fewer spindles, a flatter spectrum. So a model can **predict a person's age from their night**. The gap between predicted and true age is called the *brain age gap*, and it is studied as a marker of brain health.

You get **all the features** of the week in one table (`precomputed/cohort_features.csv`, one row per night, about 230 features). We use one model, a **Random Forest**, and start with a **weak starter set** of features (eight basic sleep statistics) that you will **improve**. The point is not the best model, it is to **evaluate honestly**, and to see that choosing features is also a form of fitting. You will:
1. set a **baseline** (what if we always predict the mean age?)
2. fit a first model, score it with **cross-validation by person**, and **check that no person is in two folds**
3. see what happens when you split **by night** instead (leakage)
4. **tune and score in one go** with *nested* cross-validation
5. **improve the feature set**, and check it on 16 people that were locked away
6. (extra) test whether a model trained on night 1 works on **night 2**

**Course rules for this notebook**
- The unit is the **person**. 75 of the 78 people have two nights, and the two nights of one person look alike. Both nights of a person must always be on the **same side** of a split.
- Anything that learns from data (filling missing values, choosing hyperparameters, selecting features) is **fitted on the training part only**. We put it all in a `Pipeline` so that this cannot go wrong.
- Metric: **MAE** in years (easy to read) and R².

In [1]:
%matplotlib inline
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, KFold, cross_val_predict
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")
RANDOM_STATE = 0

## 1 · The table, and a starter set of features

One row per night. `age` is what we predict (the **target**); `subject` tells us who the night belongs to.

**Locked test people.** Before we look at anything, we set aside **16 people (20 %)**. They are in the table `df_locked` and **nobody touches them until Exercise 5.5**. Everything else (62 people, `df`) is for developing and comparing. Why: in Section 5 you will try many feature sets and keep the best. Each try is a peek at the scores, so the best score after many tries is too optimistic. The locked people give one score that was not used to choose anything.

**Starter features.** The table has about 230 features. We start with only **eight basic sleep statistics**: total sleep time, sleep efficiency, sleep onset latency, WASO and the percentage of each stage. That is what a clinical sleep report gives. It is not optimal, and improving it is your job in Section 5.

**Missing values** are real: a person with no N3 has no N3 band power. We don't drop those nights (we have few), we fill the gaps **inside the pipeline**. (Why they are missing: Exercise 5.6.)

In [2]:
df_all = pd.read_csv("../precomputed/cohort_features.csv")

# Lock away 20 % of the PEOPLE (both nights of each), once, with a fixed seed
dev_idx, locked_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
                           .split(df_all, groups=df_all["subject"]))
df = df_all.iloc[dev_idx].reset_index(drop=True)
df_locked = df_all.iloc[locked_idx].reset_index(drop=True)
print(f"development: {len(df)} nights, {df['subject'].nunique()} people | locked: {len(df_locked)} nights, {df_locked['subject'].nunique()} people")

# All usable features: everything numeric except identifiers and the target. We leave out the columns that compare
# YASA with the expert (yasa_accuracy, yasa_kappa, yasa_F1_*): they need an expert hypnogram, and they are not a property of the night.
drop = ["code", "subject", "night", "age", "sex", "ART", "UNS"]
drop += [c for c in df.columns if c.startswith(("yasa_accuracy", "yasa_kappa", "yasa_F1"))]
ALL_FEATURES = df.drop(columns=drop).select_dtypes("number").columns.tolist()

STARTER = ["TST", "SE", "SOL", "WASO", "%N1", "%N2", "%N3", "%REM"]

X = df[STARTER]
y = df["age"].to_numpy()
groups = df["subject"].to_numpy()  # who each night belongs to
print(len(ALL_FEATURES), "features available,", len(STARTER), "in the starter set")

development: 121 nights, 62 people | locked: 32 nights, 16 people
230 features available, 8 in the starter set


## 2 · A baseline, and the first honest score

Any model needs a **baseline**, or "MAE = 14 years" means nothing. The simplest one always predicts the training mean.

We score with **cross-validation by person**: `GroupKFold` makes 5 folds so that all nights of a person fall in the same fold. In each round, the model is fitted on 4 folds and predicts the 5th.

In [3]:
def scores(y_true, y_pred):
    return {"MAE": round(mean_absolute_error(y_true, y_pred), 2), "R2": round(r2_score(y_true, y_pred), 2)}


outer = GroupKFold(n_splits=5)

baseline = DummyRegressor(strategy="mean")
pred_baseline = cross_val_predict(baseline, X, y, groups=groups, cv=outer)
print("baseline (mean age):", scores(y, pred_baseline))

baseline (mean age): {'MAE': 18.58, 'R2': -0.0}


**Exercise 2.1 — a first model.** Build a **pipeline** with two steps: fill missing values with the median (`SimpleImputer(strategy="median")`) and a `RandomForestRegressor(n_estimators=200, max_features=0.3, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)`. Score it with the same `outer` split.

Why a pipeline? `cross_val_predict` refits the *whole* pipeline on each training part. The median used to fill gaps is then computed without the test fold. (A forest doesn't need scaled features. With a model that does, a `StandardScaler` would go into the pipeline too, for the same reason.)

In [4]:
forest = make_pipeline(SimpleImputer(strategy="median"),
                       RandomForestRegressor(n_estimators=200, max_features=0.3, min_samples_leaf=3,
                                             random_state=RANDOM_STATE, n_jobs=-1))
pred_forest = cross_val_predict(forest, X, y, groups=groups, cv=outer)
print("random forest, starter set:", scores(y, pred_forest))

random forest, starter set: {'MAE': 14.39, 'R2': 0.34}


*Answer* (MAE, R²)
Baseline: 18.6 years, R² 0. Random Forest on the starter set: 14.4 years, R² 0.34. Eight basic sleep statistics carry some information about age, but not much. The baseline MAE of 18.6 is the mean absolute deviation of ages in this cohort (25–101 years, a wide and rather flat distribution).

**Exercise 2.2 — no person in two folds.** `GroupKFold` is supposed to keep each person on one side, but **never trust, check**. Write a function `overlap(cv, X, y, groups)` that returns, for each fold of `cv`, the **number of people that appear in both the training and the test part**. Run it on `outer`: you need `[0, 0, 0, 0, 0]`.

Then, as an assert that stops the notebook if it ever fails, `assert sum(...) == 0`. Put such a check in every notebook where you split people.

In [5]:
def overlap(cv, X, y, groups):
    """For each fold: how many people are in both the training and the test part?"""
    return [len(set(groups[train]) & set(groups[test])) for train, test in cv.split(X, y, groups)]


print("overlap per fold:", overlap(outer, X, y, groups))
assert sum(overlap(outer, X, y, groups)) == 0, "a person is in the training AND the test part!"

overlap per fold: [0, 0, 0, 0, 0]


## 3 · What a wrong split does

Now do it the wrong way on purpose: a plain `KFold` with shuffling splits the **nights** at random, so the two nights of the same person often end up on both sides (one in training, one in test).

**Exercise 3.1.** Score the same forest with `KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)` and compare it with the split by person. Do it for the **starter set** and for **all features**: how much a leak helps depends on how many features the model has to memorise people with. Then use your `overlap` function on the leaky split.

In [6]:
leaky = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = {}
for name, features in [("starter (8 features)", X), ("all features (230)", df[ALL_FEATURES])]:
    rows[name + ", split by person"] = scores(y, cross_val_predict(forest, features, y, groups=groups, cv=outer))
    rows[name + ", split by night"] = scores(y, cross_val_predict(forest, features, y, cv=leaky))
print(pd.DataFrame(rows).T)

print("people in both training and test, per fold:", overlap(leaky, X, y, groups))

                                         MAE    R2
starter (8 features), split by person  14.39  0.34
starter (8 features), split by night   14.54  0.33
all features (230), split by person    10.87  0.65
all features (230), split by night      9.76  0.71
people in both training and test, per fold: [19, 19, 16, 18, 20]


*Answer*
With the 8 starter features the split by night makes no difference (14.5 vs. 14.4 years). With all 230 features the gap is large: 9.8 vs. 10.9 years, R² 0.71 vs. 0.65. The model has enough features to recognise the person from the other night. `overlap` shows 16–20 people shared per fold on the leaky split (the exact numbers depend on the seed), and 0 on `GroupKFold`. **A better score from a worse split is not a better model**, and the leak grows with the number of features, so you can't judge it from a simple model. (For a neural network on raw epochs it can be much larger.)

**Question 3.** The *same* mistake exists at the epoch level, if you split 30-s epochs at random. Why is it much worse there than here?
Neighbouring epochs of one night are almost the same signal, so almost every test epoch has near-copies in training: the model can recognise the person instead of learning about age.

## 4 · Tuning: nested cross-validation

The forest settings above were a guess. To choose them we need to try several and keep the best. But if we pick the best on **the same folds we report**, the score is optimistic: we selected the winner by looking at the test folds.

**Nested cross-validation** keeps the two jobs apart:
- the **inner loop** (on the training part only) chooses the hyperparameters: `GridSearchCV` with its own `GroupKFold`
- the **outer loop** scores the chosen model on people it has never seen

```
outer fold 1: train = people A  → inner CV on A picks the settings → refit on A → score on test people (never used)
outer fold 2: ...
```
The reported number is the average over the outer test folds. The settings **chosen** in each outer fold can differ: that's fine, we are evaluating the *procedure* "tune, then fit".

In [7]:
def nested_cv(make_search, X, y, groups, n_outer=5):
    """Nested CV by person. `make_search()` returns a fresh GridSearchCV. Returns predictions and the chosen settings."""
    pred = np.zeros(len(y))
    chosen = []
    for train, test in GroupKFold(n_splits=n_outer).split(X, y, groups):
        search = make_search()
        search.fit(X.iloc[train], y[train], groups=groups[train])  # inner folds are also by person
        pred[test] = search.predict(X.iloc[test])
        chosen.append(search.best_params_)
    return pred, chosen

**Exercise 4.1 — tune the forest.** Complete the search: imputer + `RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)`; try `max_features` in `[0.3, 0.6]` (the share of features each split may look at) and `min_samples_leaf` in `[1, 3]` (the smallest leaf; larger leaves = a smoother model); an inner `GroupKFold(4)` and MAE as the score (`scoring="neg_mean_absolute_error"`).

The name of a pipeline parameter is `<step name>__<parameter>`: for the forest in a `make_pipeline` it is `randomforestregressor__max_features`.

In [8]:
def make_rf_search():
    pipe = make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))
    grid = {"randomforestregressor__max_features": [0.3, 0.6],
            "randomforestregressor__min_samples_leaf": [1, 3]}
    return GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=4), scoring="neg_mean_absolute_error")


pred_nested, chosen = nested_cv(make_rf_search, X, y, groups)
print("random forest, nested CV, starter set:", scores(y, pred_nested))
for settings in chosen:
    print(settings)

random forest, nested CV, starter set: {'MAE': 14.45, 'R2': 0.34}
{'randomforestregressor__max_features': 0.3, 'randomforestregressor__min_samples_leaf': 3}
{'randomforestregressor__max_features': 0.6, 'randomforestregressor__min_samples_leaf': 3}
{'randomforestregressor__max_features': 0.3, 'randomforestregressor__min_samples_leaf': 3}
{'randomforestregressor__max_features': 0.3, 'randomforestregressor__min_samples_leaf': 3}
{'randomforestregressor__max_features': 0.3, 'randomforestregressor__min_samples_leaf': 3}


*Answer*
About 14.5 years, R² 0.34: no better than the guess, and the chosen settings are mostly the same one (`min_samples_leaf=3`). With eight features, there is little to tune: the limit is the **information in the features**, not the settings. This is the number to report for the starter set: it was never used to choose anything.

**Exercise 4.2 — the inner folds too.** The inner `GroupKFold(4)` also splits people. Use `overlap` to check that the inner folds have no person in both parts, for each of the 5 outer training parts (`GroupKFold(5).split(X, y, groups)`: apply `overlap` to `X.iloc[train]`, `y[train]`, `groups[train]` with a `GroupKFold(4)`). Print the result.

In [9]:
for k, (train, test) in enumerate(outer.split(X, y, groups)):
    inner_overlap = overlap(GroupKFold(n_splits=4), X.iloc[train], y[train], groups[train])
    print(f"outer fold {k}: inner overlaps {inner_overlap}")
    assert sum(inner_overlap) == 0

outer fold 0: inner overlaps [0, 0, 0, 0]
outer fold 1: inner overlaps [0, 0, 0, 0]
outer fold 2: inner overlaps [0, 0, 0, 0]
outer fold 3: inner overlaps [0, 0, 0, 0]
outer fold 4: inner overlaps [0, 0, 0, 0]


## 5 · Improve the features

The starter set is weak on purpose. Your task: **find a better set of features**, using what you know about how sleep changes with age (less deep sleep and slow waves, fewer spindles, a flatter spectrum, more fragmented sleep, ...).

**Rules**
1. Every try is scored with the **same nested CV** (`evaluate` below, on the 62 development people).
2. Choose features for a **reason** you can write down. Don't just add everything that raises the score.
3. Count your tries. After 20 tries the best score is partly luck, and only the locked people (5.5) tell you how much.

In [10]:
def evaluate(features, make_search=make_rf_search):
    """Nested CV by person on the development people, for a list of feature names."""
    pred, _ = nested_cv(make_search, df[features], y, groups)
    return scores(y, pred)


def family(name):
    if name.startswith("yasa_"):
        return "YASA staging"
    if name.startswith(("sp_", "sw_")):
        return "spindles / slow waves"
    if name.split("_")[0] in ("delta", "theta", "alpha", "sigma", "beta", "gamma"):
        return "band power"
    return "sleep statistics"


fam = pd.Series({f: family(f) for f in ALL_FEATURES})
print(fam.value_counts())
print("starter:", evaluate(STARTER))

band power               120
YASA staging              46
sleep statistics          38
spindles / slow waves     26
Name: count, dtype: int64
starter: {'MAE': 14.45, 'R2': 0.34}


**Question 5.1.** The table has about 230 features for 62 people. Why is "use them all" not automatically the answer?

*Answer*
With more features than people, a model can find patterns that are noise: it fits the training ages and fails on new people. Many features are also near-copies of each other (relative power of neighbouring bands, the same measure on two channels), so the number of independent features is much smaller than 230, and the useful ones are diluted among many useless ones. A forest is fairly robust to this (each split looks at a random share of the features), but not immune. At the other extreme, eight features can't capture the age effects on spectrum, spindles or slow waves.

**Exercise 5.2 — one family at a time.** Add **one family** of features to the starter set (`STARTER + list(fam.index[fam == name])`) and score it, for each of the four families. Also score each family **alone** and **all features together**. Which family helps most? Does everything together beat the best single addition?

In [11]:
rows = {}
for name in fam.unique():
    members = list(fam.index[fam == name])
    rows["starter + " + name] = evaluate(STARTER + [f for f in members if f not in STARTER])
    rows[name + " alone"] = evaluate(members)
rows["all features"] = evaluate(ALL_FEATURES)
pd.DataFrame(rows).T

,MAE,R2
starter + sleep statistics,13.66,0.40
sleep statistics alone,13.73,0.39
starter + band power,12.31,0.54
band power alone,12.98,0.50
starter + spindles / slow waves,10.95,0.63
spindles / slow waves alone,11.16,0.62
starter + YASA staging,12.48,0.53
YASA staging alone,12.65,0.53
all features,10.76,0.66


*Answer* (nested forest, MAE / R² on the 62 development people)

| set | MAE | R² |
|---|---|---|
| starter (8) | 14.5 | 0.34 |
| starter + sleep statistics | 13.7 | 0.40 |
| starter + band power | 12.3 | 0.54 |
| starter + YASA staging | 12.5 | 0.53 |
| starter + spindles / slow waves | 10.9 | 0.63 |
| spindles / slow waves **alone** (26 features) | 11.2 | 0.62 |
| all 230 features | 10.8 | 0.66 |

Spindle and slow-wave features carry the most information about age (26 features alone get close to the full table). Everything together is only slightly better than the best single family: the families are redundant (delta power, %N3 and slow-wave density all measure the age-related loss of deep sleep). Your numbers can differ by a few tenths of a year with other settings.

**Exercise 5.3 — your own feature set.** Build a list `MY_FEATURES` from the starter set plus the features you think matter, with a reason for each (write it as a comment). You can pick whole families, one stage (e.g. only N3), one channel (only `Fpz`), or single columns. Score it with `evaluate`.

Hints: the column names follow `<band>_<abs|rel>_<stage>_<channel>` (e.g. `delta_rel_N3_Fpz`) and `sp_`/`sw_` for spindles and slow waves. Absolute power depends on the skull and the electrode contact; relative power is less sensitive to that.

In [12]:
# Reasons: spindle and slow-wave density drop with age (Session 7); relative power in N2 and N3 reflects the age
# loss of slow waves and the slower spectrum, and is less affected by skull thickness than absolute power.
MY_FEATURES = (STARTER
               + [f for f in ALL_FEATURES if family(f) == "spindles / slow waves"]
               + [f for f in ALL_FEATURES if "_rel_" in f and f.split("_")[2] in ("N2", "N3") and f.endswith("Fpz")])
print(len(MY_FEATURES), "features:", evaluate(MY_FEATURES))

46 features: {'MAE': 10.88, 'R2': 0.65}


**Exercise 5.4 — let the pipeline choose.** Choosing by hand is one option; the other is to let the model choose, **inside the pipeline**, so that the choice is made on the training part only. `SelectKBest(f_regression, k=...)` keeps the `k` features most correlated with age; `k` is then tuned in the inner loop together with the forest's `min_samples_leaf`.

**Careful:** picking the 30 best features on the *whole* table and then running cross-validation is a classic leak: the selection has already seen the test people.

Complete the search over `k` in `[10, 30, 60]` and `min_samples_leaf` in `[1, 3]`, on all features.

In [13]:
def make_kbest_search():
    pipe = make_pipeline(SimpleImputer(strategy="median"), SelectKBest(f_regression),
                         RandomForestRegressor(n_estimators=200, max_features=0.3, random_state=RANDOM_STATE, n_jobs=-1))
    grid = {"selectkbest__k": [10, 30, 60], "randomforestregressor__min_samples_leaf": [1, 3]}
    return GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=4), scoring="neg_mean_absolute_error")


print("all features + SelectKBest:", evaluate(ALL_FEATURES, make_kbest_search))

all features + SelectKBest: {'MAE': 11.22, 'R2': 0.6}


**Exercise 5.5 — the locked people (once).** Now fit the **whole procedure** (tuning included) on all 62 development people and score the 16 locked people, for the starter set, your set and the automatic selection. Do this **once**. If you go back and change `MY_FEATURES` after seeing this, the locked people are no longer locked.

In [14]:
def locked_score(features, make_search=make_rf_search):
    search = make_search()
    search.fit(df[features], y, groups=groups)  # tuning by inner CV on the 62 development people
    return scores(df_locked["age"], search.predict(df_locked[features]))


pd.DataFrame({"starter": locked_score(STARTER),
              "my features": locked_score(MY_FEATURES),
              "all + SelectKBest": locked_score(ALL_FEATURES, make_kbest_search)}).T

,MAE,R2
starter,12.41,0.35
my features,10.77,0.35
all + SelectKBest,9.92,0.52


*Answer* (16 locked people, 32 nights; tuning on the 62 development people)

| set | development MAE | locked MAE | locked R² |
|---|---|---|---|
| starter | 14.5 | 12.4 | 0.35 |
| my features (46: starter + spindles/SW + relative power N2/N3 Fpz) | 10.9 | 10.8 | 0.35 |
| all + SelectKBest | 11.2 | 9.9 | 0.52 |

Both improvements hold up on people that were not used for any choice. The hand-picked set and the automatic selection are within the noise of each other (10.8 vs. 9.9 years; 16 people is a small sample, so differences of 1 year mean little). R² is lower for the hand-picked set because 16 people cover a narrower range of ages.

**Question 5.5.** Your development score and the locked score differ. Which one do you report, and what does it tell you if your *hand-picked* set does clearly worse on the locked people than on the development people?

*Answer*
Report the locked score, with its caveat: 16 people is a small, noisy sample, so the interval is wide (several years). A drop from development to locked is expected, because the set was chosen by looking at the development scores; the more sets tried, the larger the drop. A large drop is a sign that the choice was fitted to the development people.

**Exercise 5.6 — why are features missing?** Some nights have missing values (`NaN`). They are not random failures. Find out the reason for three groups of columns:
1. `delta_abs_N3_Fpz` (all N3 band power): list the nights where it is missing, with `age`, `N3` (minutes of N3 by the expert) and `clean_min_N3`.
2. `sw_ptp_Fpz` (average slow-wave size): list the nights where it is missing, with `age`, `clean_min_N2`, `sw_density_N2_Fpz`.
3. `sw_ptp_Pz`: how many nights are missing, and what is `sw_density_N2_Pz` there? Is missingness related to age?

Use `df_all` (all 153 nights).

In [15]:
miss_n3 = df_all["delta_abs_N3_Fpz"].isna()
print("1. N3 band power missing:", miss_n3.sum(), "nights; expert N3 minutes in those nights:", df_all.loc[miss_n3, "N3"].unique())
print(df_all.loc[miss_n3, ["code", "age", "N3", "clean_min_N3"]].sort_values("age").head())

miss_sw = df_all["sw_ptp_Fpz"].isna()
print("2. slow-wave size (Fpz) missing:", miss_sw.sum(), "nights")
print(df_all.loc[miss_sw, ["code", "age", "clean_min_N2", "sw_density_N2_Fpz"]])

miss_pz = df_all["sw_ptp_Pz"].isna()
print("3. slow-wave size (Pz) missing:", miss_pz.sum(), "nights; slow-wave density N2 there:", df_all.loc[miss_pz, "sw_density_N2_Pz"].unique())
print("median age, missing vs. present:", df_all.loc[miss_pz, "age"].median(), df_all.loc[~miss_pz, "age"].median())
print("share of nights with at least one missing feature, by age group:")
df_all.groupby(pd.cut(df_all["age"], [20, 40, 60, 80, 105]))[ALL_FEATURES].apply(lambda g: g.isna().any(axis=1).mean()).round(2)

1. N3 band power missing: 18 nights; expert N3 minutes in those nights: [0.]
      code  age   N3  clean_min_N3
40  SC4202   51  0.0           0.0
67  SC4341   54  0.0           0.0
68  SC4342   54  0.0           0.0
63  SC4321   57  0.0           0.0
65  SC4331   60  0.0           0.0
2. slow-wave size (Fpz) missing: 8 nights
       code  age  clean_min_N2  sw_density_N2_Fpz
46   SC4232   50          98.5                0.0
67   SC4341   54         270.5                0.0
68   SC4342   54         292.5                0.0
123  SC4641   85         188.5                0.0
124  SC4642   85         255.0                0.0
137  SC4731   97         205.0                0.0
139  SC4741   92         253.0                0.0
140  SC4742   92         169.0                0.0
3. slow-wave size (Pz) missing: 47 nights; slow-wave density N2 there: [0.]
median age, missing vs. present: 71.0 54.0
share of nights with at least one missing feature, by age group:


age
(20, 40]     0.08
(40, 60]     0.37
(60, 80]     0.36
(80, 105]    0.78
dtype: float64

*Answer*
1. **No N3 at all.** In all 18 nights with missing N3 band power the expert scored **0 minutes of N3** (ages 51–97, median 87). With no N3 epochs there is nothing to average; the feature is *undefined*, not zero. The same nights have no N3 spindle or slow-wave density.
2. **No slow waves found.** The slow-wave size, slope, frequency and coupling are *averages over detected slow waves*. In 8 nights (median age 85) the detector found none (density 0 in N2, and no N3 to look in), so the average doesn't exist. The same happens to the spindle amplitude, frequency and duration in the 2 nights (age 89) without spindles.
3. **Small waves on the occipital channel.** On Pz-Oz 47 nights have no detected slow waves (density 0 in N2): slow waves are probably smaller on the occipital channel and don't reach YASA's default amplitude threshold (a hypothesis: check it by lowering the threshold, as in Session 7). Missingness goes with age (median 71 vs. 54 years); the share of nights with at least one missing feature goes from 8 % (under 40 y) to 37 %, 36 % and 78 % (over 80 y).

So **missingness is information**: "no N3" and "no slow waves" are themselves signs of an old brain, and the missing values are concentrated in the oldest people. The median imputer replaces them by the typical training value, which hides this. A model could keep it with `SimpleImputer(strategy="median", add_indicator=True)`, which adds a 0/1 column for each feature with gaps (you can try it as an extra: `make_pipeline(SimpleImputer(strategy="median", add_indicator=True), ...)`). A forest can also split on other features that carry the same information, which is why the effect is small here.

## 6 · Extra: does it work on the second night?

The cohort has two nights per person (night 2 is the person's second recording; a first night in a new setting is known to differ from later ones, so night 1 and night 2 are not interchangeable). Different ways to split the data answer **different questions**:

| split | question it answers |
|---|---|
| (a) train on night 1, test on night 2 of the **same people** | can we recognise this person's age from a second recording? (the model has seen the person) |
| (b) train on night 1 of some people, test on night 2 of **other people** | does the model work on new people *and* a new night? |
| (c) train on night 2, test on night 1 of **other people** | the same, in the other direction (first-night effect?) |

**Exercise 6.1.** Use a forest (`forest`, settings of Exercise 2.1) with your feature set `MY_FEATURES`, on the development people. For (b) and (c), draw 5 random splits of **people** into 80 % train and 20 % test (`GroupShuffleSplit`) and average the MAE: one split of 13 people is too noisy to compare. Use `overlap` to check each split.

In [ ]:
night = df["night"].to_numpy()
X = df[MY_FEATURES]


def fit_score(train_mask, test_mask, model=forest):
    model.fit(X[train_mask], y[train_mask])
    return mean_absolute_error(y[test_mask], model.predict(X[test_mask]))


# (a) same people, other night
a = {f"night {tr} -> night {te}": fit_score(night == tr, night == te) for tr, te in [(1, 2), (2, 1)]}


# (b), (c) new people, other night
def new_people(train_night, test_night, n_splits=5):
    maes = []
    splitter = GroupShuffleSplit(n_splits, test_size=0.2, random_state=RANDOM_STATE)
    assert sum(overlap(splitter, X, y, groups)) == 0
    for train_p, test_p in splitter.split(X, y, groups):
        train_mask = np.isin(np.arange(len(y)), train_p) & (night == train_night)
        test_mask = np.isin(np.arange(len(y)), test_p) & (night == test_night)
        maes.append(fit_score(train_mask, test_mask))
    return np.mean(maes), np.std(maes)


b = {f"new people, night {tr} -> night {te}": new_people(tr, te) for tr, te in [(1, 2), (2, 1)]}
c = {f"new people, night {n} -> night {n}": new_people(n, n) for n in [1, 2]}  # control: same night
print({k: round(v, 2) for k, v in a.items()})
for k, (m, s) in {**b, **c}.items():
    print(f"{k}: MAE {m:.2f} ± {s:.2f}")

*Answer* (forest, MY_FEATURES; 5 random splits, 13 test people each, so every number has an SD of 1–2 y)
(a) Same people, other night: MAE 9.3 (night 1 → 2) and 10.7 (night 2 → 1), lower than any split with new people, because the model has seen the person. (b) New people, other night: 11.2 ± 1.1 and 12.1 ± 0.9. (c) Same-night controls: 11.8 ± 0.5 (night 1) and 11.9 ± 1.0 (night 2). **We can't say that the night matters**: the other-night scores are as good as the same-night controls, and the differences are within the spread of the splits. What we can say is that the person-level leak in (a) is worth about 1–2.5 years. For an effect you can believe, use more splits (`RepeatedGroupKFold`) and report differences with their SD.

**Question 6.** Why is (a) the *least* informative split for a "brain-age" application? What would you need to claim that the model works on a **new person recorded once**?
(a) re-uses the person: it measures how stable a person's features are across two nights (spindle density correlates at 0.92 between a person's two nights, Session 7), not generalisation. A new person recorded once is (b)/(c) or the subject-wise CV of Section 4.

## Wrap-up

- The headline number is the **nested, subject-wise** MAE, compared with the **mean-age baseline**.
- **Check** that no person is in two folds (`overlap`), and keep the check in the notebook.
- A random split of nights (or epochs!) gives a better number and a worse model, and the leak grows with the number of features.
- Hyperparameters are chosen **inside** the training part; imputing and feature selection are in the **pipeline**.
- **Choosing features is fitting too.** Every set you try and compare is a peek at the scores: keep people locked away for one final check, or put the selection inside the pipeline.
- Features beat settings: tuning the forest changed little, the feature set changed the error from 14.5 to about 11 years.
- Missing values are not random: here they mean "no N3" or "no slow waves", which is a sign of age.

**Extra ideas:** `add_indicator=True` in the imputer; average the two nights of a person before the final score (one prediction per person); `RepeatedGroupKFold` to see the spread of the score; `permutation_importance` on the held-out folds; another model (Ridge, LightGBM) with the same pipeline and the same folds; add `sex` as a feature.

Deep learning on the raw signal: see the lecture. It needs the same rules (split by person, tune inside the training part, a baseline), and much more data than 78 people.